# Week 2: Dynamic Programming Methods

Craig Rudman  
Regis University  
MSDS684 Reinforcement Learning  
Prof. Mike Busch  

## Section 1: Project Overview 

This lab investigates the prediction and control problems in reinforcement learning through dynamic programming, specifically asking: given a known model, how can an agent compute optimal value functions and policies, and how do different algorithms compare in convergence behavior?

The core concepts explored come directly from Sutton & Barto, Chapter 4. Solving the optimization problem requires two fundamental components: policy evaluation, which computes the state-value function for a given policy, and policy improvement, which produces a better policy by acting greedily with respect to the current value function. The policy improvement theorem guarantees that this step yields a policy that is at least as good as the previous one. Both policy iteration and value iteration combine these components, but differ in how they structure the interaction. Policy iteration runs evaluation to full convergence before each improvement step, while value iteration truncates evaluation to a single sweep and improves immediately. Both fall under the generalized policy iteration (GPI) framework, which describes evaluation and improvement as two interacting processes that converge toward optimality regardless of how they are interleaved.

Theoretically, these algorithms are grounded in the contraction mapping properties of the Bellman operators. Each sweep of the value function brings the estimate closer to the true value, with convergence guaranteed under the assumption of a finite model with known transition dynamics. This is dynamic programming's defining requirement and its primary limitation: the agent must have access to all of the known transition dynamics of the model represented by P(s',r|s,a). The lab reinforces this by requiring explicit construction or extraction of those probabilities.

The environment is a custom GridWorld built with Gymnasium's API. The state space is discrete and two-dimensional. The action space is also discrete with four actions: up, down, left, and right. The reward structure is configurable; a typical setup assigns a small negative reward per step to encourage efficiency, with a larger positive reward at the goal state and possible penalties for obstacles. Episodes terminate when the agent reaches the goal or a maximum step count. The environment supports both deterministic transitions and stochastic transitions, allowing comparison across conditions. The same algorithms are also applied to Gymnasium's FrozenLake-v1.

I hypothesize that value iteration will converge in fewer iterations than policy iteration under most conditions, since it does not require full convergence of the value function before improving the policy. Additionally, stochastic environments should require more iterations for both algorithms, as the added uncertainty broadens the value landscape and slows the contraction toward optimal values. Finally, in-place updates should converge faster than synchronous versions, since they immediately propagate new information within a single sweep.

## Section 2: Deliverables

GitHub Repository: https://github.com/craig-rudman/MSDS684/tree/main/W2

### 2.1 Implementation Summary

We implemented policy iteration and value iteration, each in synchronous and in-place variants, and applied all four solvers to two environments: a custom Gymnasium-compatible GridWorld and Gymnasium's FrozenLake-v1. Development testing used a simple 4x4 deterministic grid with one obstacle, a step penalty of -1, and a goal reward of +1.0. Acceptance testing scaled to an 8x8 grid with five obstacles, mixed rewards (+10.0 goal, -5.0 penalty state, -0.04 step cost), under both deterministic and stochastic dynamics (80% intended, 10% each perpendicular). FrozenLake was evaluated on the 8x8 map with slippery enabled and disabled. All solvers shared gamma = 0.99 and theta = 1e-8, centralized in a single configuration cell. An ExperimentRunner orchestrated all four variants, collecting iteration counts and converged value functions for comparison.

### 2.2 Key Results & Analysis

I compared the performance of value iteration and policy iteration, implemented synchronously and in-place, in deterministic environments and stochastic environments, across grid sizes 4, 8, 16, 32, and 64.

#### Key Results:

- Value Iteration (VI) dominates policy iteration (PI) on the time to convergence in wall-clock units in both environments, and the gap grows with grid size.

- Stochasticity is what creates the interesting tradeoffs. Under deterministic dynamics, all variants converge in exactly 2N - 1 iterations -- synchronous vs in-place and VI vs PI are indistinguishable in iteration count. Stochasticity breaks this symmetry by making value propagation diffuse across successor states.

- In-place helps VI under stochasticity because within-sweep propagation cuts through the slow diffusion of stochastic updates. Under deterministic dynamics this advantage disappears.

- In-place has mixed effects on PI. It speeds up inner evaluation but can increase outer cycles (noisier signal for policy improvement). The two effects roughly cancel on wall clock under stochasticity.  

#### Analysis: 

##### Deterministic environments

Implementation and algorithm exhibit similar behavior when environment dynamics are deterministic, converging in the same number of iterations.

<figure>
    <img src="../img/all_variants_iterations_deterministic.png" alt="Iterations to convergence in deterministic environments" width='100%'>
    <figcaption>Figure 1: Iterations to convergence by algorithm and implementation in deterministic environments.</figcaption>
</figure>

In deterministic environments, there is no branching over successor states, so the value iteration and the policy iteration effectively reduce to the same values rather than weighted sums over multiple possible successors.

In terms of time to converge in wall-clock units, value iteration demonstrates greater efficiencies, with the gap widening as grid size increases.

<figure>
    <img src="../img/all_variants_wallclock_deterministic.png" alt="Time to convergence in deterministic environments" width='100%'>
    <figcaption>Figure 2: Time to convergence in wall-clock units by algorithm and implementation in deterministic environments.</figcaption>
</figure>

We should also note that, under deterministic dynamics, the synchronous implementations of both algorithms prove to be more efficient than the in-place implementations in wall-clock units.

##### Stochastic environments

When environment dynamics are stochastic, policy iteration converges in significantly fewer iterations than value iteration.

<figure>
    <img src="../img/all_variants_iterations_stochastic.png" alt="Iterations to convergence in stochastic environments" width='100%'>
    <figcaption>Figure 3: Iterations to convergence by algorithm and implementation in stochastic environments.</figcaption>
</figure>

Value iteration performs one sweep per iteration, gradually improving policy one iteration at a time. Policy iteration runs multiple sweeps per iteration fully solving for the value function for the current policy, thus taking bigger strides when improving policies.

This comes at a cost. Policy iteration takes significantly more time to converge than value iteration, and this gap widens dramatically as grid size increases.

<figure>
    <img src="../img/all_variants_wallclock_stochastic.png" alt="Time to convergence in stochastic environments" width='100%'>
    <figcaption>Figure 4: Time to convergence in wall-clock units by algorithm and implementation in stochastic environments.</figcaption>
</figure>

##### Visualizing Convergence

Figure 5 demonstrates how value iteration converges on the optimal value function for a 4 x 4 deterministic grid.

<figure>
    <img src="../img/value_function_contact_sheet.png" alt="Value iteration convergence heatmap" width='100%'>
    <figcaption>Figure 5: Value function convergence on a 4x4 grid in eight iterations.</figcaption>
</figure>

We can see how the value of each state is determined starting with the goal (the terminal state) and working backwards. States become less valuable with distance from the goal.

Figure 6 shows how policy iteration finds the optimal policies for a 4 x 4 deterministic grid.

<figure>
    <img src="../img/policy_iteration_contact_sheet.png" alt="Value iteration convergence heatmap" width='100%'>
    <figcaption>Figure 6: Policy optimization on a 4x4 grid in eight iterations.</figcaption>
</figure>

Once again, we can see how the policy iteration algorithm optimizes policy working from the goal backwards.

##### Conclusion

My hypothesis was incorrect on a number of counts. 

I misunderstood the nature of iterations, confusing them with sweeps. Value iteration takes one sweep per iteration, but policy iteration takes as many sweeps as necessary to fully evaluate the current policy before improving it. This distinction explains why policy iteration in stochastic environments takes fewer iterations to converge, but takes more time to converge.

I hypothesized that stochastic environments would require more iterations than deterministic environments for both algorithms. While true for value iteration, it was not true for policy iteration. Stochastic transitions sometimes leak value information to distant states, allowing each improvement to capitalize on those opportunities.

Finally, I hypothesized that in-place updates should converge faster than synchronous versions. The results here are conditional. In-place updates consistently help value iteration under stochasticity by propagating information faster within each sweep. For policy iteration, the effects are mixed, perhaps with faster inner evaluation but noisier policy improvement, requiring more outer cycles.

<figure>
    <img src="../img/convergence_curves_vi.png" alt="Comparing convergence curves for value iteration" width='100%'>
    <figcaption>Figure 7: Convergence curve for value iteration on a 4x4 deterministic grid.</figcaption>
</figure>

## Section 3: AI Use Reflection

### 3.1 Initial Interaction

I asked Claude to stub out a reinforcement learning lab covering GridWorld environments, policy iteration, value iteration, visualization, and CLI execution. Claude created class stubs across four source modules and a pytest suite with 47 tests, all raising NotImplementedError. I then requested a TDD approach, and Claude organized the tests into a red-green workflow for iterating through each TODO item. All source code is class-encapsulated and runnable from the CLI.

### 3.2 Iteration Cycle

#### 3.2.1 Iteration One

We ran the 20-test GridWorld suite (all red). Claude implemented the 7 stub methods in gridworld.py -- coordinate helpers, deterministic/stochastic transition model, and Gymnasium reset/step/render. Tests went green. Claude wired the first notebook TODO cell; I verified it from the CLI.

The environment now supports configurable geometry, stochastic transitions (80/10/10 perpendicular split), and exposes the full transition model P(s',r|s,a), unblocking both DP solvers next.

#### 3.3.2 Iteration Two

Ran 23 tests (all red). Claude implemented evaluate_sync, improve, and solve in dp_solvers.py. A terminal-state bug surfaced where V(goal)=1.0 instead of 0.0; fixed by skipping terminal states during evaluation. All 5 sync PI tests went green. Claude wired the solver into the notebook and backfilled validation cells for three prior TODOs, flipping tags from # TODO to # DONE. Updated the README status table to match.

#### 3.3.3 Iteration Three

I ran the 4 in-place policy iteration tests (all red). Claude implemented evaluate_inplace in dp_solvers.py, mirroring the sync version but updating V(s) in place so later states see fresh values within the same sweep. All 4 tests went green with no regressions. Claude wired the notebook cell with a demo and sync-vs-inplace equivalence assertion, flipped the tag to DONE, and updated the README status table.

#### 3.3.4 Subsequent Iterations



Subsequent iterations followed the same pattern... run the tests, code to the next TODO item, validate the implementation with smoke tests, fix problems (if any), re-run the suite of automated unit tests, update the README file, and commit to GitHub.

### 3.3 Critical Evaluation

I used test-driven development with unit tests. Some mistakes still slipped through:

Action to direction mapping error
The mapping didn't account for GridWorld's inverted y-axis. UP and DOWN arrows rendered backwards, caught only through visual inspection.

Plotting convergence curves
Claude initially plotted all four solver variants on one y-axis. PI deltas (~100) dwarfed VI deltas (~1-5), making VI invisible. We fixed this by splitting PI and VI into separate figures.

### 3.4 Learning Reflection

I came away with a much better understanding of the different approaches to policy optimization taken by value iteration versus policy iteration. Value iteration propogates value estimates backwards over multiple iterations to find the optimal values for each state, and then extracts the optimal policy.  Policy iteration solves for the best policy with each iteration, an arguably more direct, but computationally more intensive, process.

Working with AI tools, I deconstructed the lesson plan into a series of TODO statements, had Claude generate stubs that threw `NotImplementedErrors` and a set of unit test that would verify implementation. Then I gave Claude the following prompt:

`Let's run one test, code, debug cycle.  Start with the test suite, then let's implement the next TODO in the Lab 2 development notebook.  We will test it from the CLI.  Flip the TODO tag to DONE when the implementation has been completed and verified. If appropriate, wire it into the runner.py.  Update the README file where appropriate.  When this has all been done, draft comments for checking into git, but don't worry about taking the commit action... I'll do that manually from the MacOS GUI`.

This is a pattern I will continue to apply in the future.

## Section 4: Speaker Notes

- Problem: How do policy iteration and value iteration compare in convergence behavior across deterministic and stochastic environments, and does in-place updating consistently help?

- Method: Implemented four DP solvers (PI sync, PI in-place, VI sync, VI in-place) with shared hyperparameters (gamma=0.99, theta=1e-8), tested on custom GridWorld (4x4 through 64x64) and FrozenLake-v1 under both deterministic and stochastic dynamics.

- Design choice: Built an ExperimentRunner that orchestrates all four variants with centralized config, collecting iteration counts, wall-clock times, and converged value functions for apples-to-apples comparison.

- Key result: VI dominates PI in wall-clock time, and the gap widens with grid size. Under deterministic dynamics all four variants converge in the same number of iterations; stochasticity breaks this symmetry, with PI taking fewer iterations but far more time due to full evaluation sweeps per cycle.

- Insight: My initial hypothesis confused iterations with sweeps. PI's "fewer iterations" each contain many sweeps, making it slower overall. The iteration-vs-sweep distinction is essential for understanding DP tradeoffs.

- Challenge: Convergence curve plotting initially put PI and VI on the same y-axis, which made VI invisible since PI deltas were two orders of magnitude larger. Also caught an inverted UP/DOWN arrow bug only through visual inspection, not unit tests.

- Connection: Deterministic environments were trivially cheap to solve; adding even modest stochasticity increased compute dramatically at scale. In practice, reducing uncertainty in your system's dynamics directly translates to faster and cheaper optimization.